# Risk Analytics Platform - Validation and Testing Notebook

This notebook provides comprehensive validation and testing for all execution modes:
- **Local Mode**: Fastest development, no external services
- **Hybrid Mode**: Local Spark + remote catalog/storage
- **Docker Mode**: Full stack with all services

## Prerequisites

1. Set the execution mode as an environment variable before starting this notebook:
   ```powershell
   $env:EXECUTION_MODE = "local"  # or "hybrid" or "docker"
   ```

2. For Hybrid/Docker modes, ensure required services are running:
   - Hybrid: `docker compose up -d nessie seaweedfs`
   - Docker: `docker compose up -d`

3. Run the pipeline for the mode you want to test before executing this notebook.

## Notebook Structure

### Part 1: Environment Validation
- Check execution mode
- Verify Spark connection
- Verify catalog connection
- Verify storage connection

### Part 2: Table Structure Validation
- Check namespace creation
- Verify table existence
- Validate table schemas

### Part 3: Data Loading Validation
- Stage layer data validation
- ODS layer data validation
- Risk metrics validation

### Part 4: Data Quality Checks
- Record count validation
- Data completeness checks
- Business logic validation

### Part 5: Performance Validation
- Query performance tests
- Storage usage analysis

### Part 6: End-to-End Testing
- Run complete pipeline
- Validate final outputs
- Check metrics calculations

In [ ]:
# Import required libraries
import os
import sys
from datetime import date
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from risk_analytics.config import load_config
from risk_analytics.spark import create_spark_session

print("✓ Libraries imported successfully")

## Part 1: Environment Validation

In [ ]:
# Check execution mode
execution_mode = os.getenv("EXECUTION_MODE", "docker")
print(f"Execution Mode: {execution_mode}")

if execution_mode not in ["local", "hybrid", "docker"]:
    print(f"⚠ WARNING: Invalid execution mode '{execution_mode}'. Expected: local, hybrid, or docker")
else:
    print(f"✓ Valid execution mode: {execution_mode}")

In [ ]:
# Load configuration
config = load_config()
print("✓ Configuration loaded successfully")
print(f"\nConfiguration Details:")
print(f"  Catalog: {config.get('catalog', {}).get('name', 'unknown')}")
print(f"  Catalog Type: {config.get('catalog', {}).get('type', 'unknown')}")
print(f"  Catalog URI: {config.get('catalog', {}).get('uri', 'N/A')}")
print(f"  Storage Type: {config.get('storage', {}).get('type', 'unknown')}")
print(f"  Storage Endpoint: {config.get('storage', {}).get('endpoint', 'N/A')}")
print(f"  Spark Mode: {config.get('spark_mode', 'unknown')}")

In [ ]:
# Create Spark session
print("Creating Spark session...")
spark = create_spark_session("validation-notebook")
print("✓ Spark session created successfully")
print(f"Spark Version: {spark.version}")
print(f"Spark Master: {spark.conf.get('spark.master')}")

In [ ]:
# Test catalog connection
print("Testing catalog connection...")
try:
    spark.sql("SHOW NAMESPACES").show()
    print("✓ Catalog connection successful")
except Exception as e:
    print(f"✗ Catalog connection failed: {e}")

## Part 2: Table Structure Validation

In [ ]:
# Check namespaces based on execution mode
print("Checking namespaces...")

expected_namespaces = []
if execution_mode == "docker":
    expected_namespaces = ["risk_analytics", "risk_analytics_stage", "risk_analytics_ods"]
else:
    expected_namespaces = ["risk_analytics_stage", "risk_analytics_ods"]

namespaces_df = spark.sql("SHOW NAMESPACES")
existing_namespaces = [row.namespace for row in namespaces_df.collect()]

print(f"Expected namespaces: {expected_namespaces}")
print(f"Existing namespaces: {existing_namespaces}")

for ns in expected_namespaces:
    if ns in existing_namespaces:
        print(f"✓ Namespace '{ns}' exists")
    else:
        print(f"✗ Namespace '{ns}' missing")

In [ ]:
# Check tables in stage namespace
print("\nChecking tables in stage namespace...")
try:
    stage_tables = spark.sql("SHOW TABLES IN risk_analytics_stage")
    stage_tables.show(truncate=False)
    print(f"✓ Found {stage_tables.count()} tables in stage namespace")
except Exception as e:
    print(f"✗ Error checking stage tables: {e}")

In [ ]:
# Check tables in ODS namespace
print("\nChecking tables in ODS namespace...")
try:
    ods_tables = spark.sql("SHOW TABLES IN risk_analytics_ods")
    ods_tables.show(truncate=False)
    print(f"✓ Found {ods_tables.count()} tables in ODS namespace")
except Exception as e:
    print(f"✗ Error checking ODS tables: {e}")

In [ ]:
# Check source tables (only in Docker mode)
if execution_mode == "docker":
    print("\nChecking tables in source namespace (Docker mode only)...")
    try:
        source_tables = spark.sql("SHOW TABLES IN risk_analytics")
        source_tables.show(truncate=False)
        print(f"✓ Found {source_tables.count()} tables in source namespace")
    except Exception as e:
        print(f"✗ Error checking source tables: {e}")
else:
    print("\n⚠ Source tables not checked (only applicable in Docker mode)")

## Part 3: Data Loading Validation

In [ ]:
# Validate Stage Layer Data
print("Validating Stage Layer Data")
print("=" * 60)

stage_entities = ["customer", "asset", "collateral", "deals"]
sources = ["sourcea", "sourceb"]

for entity in stage_entities:
    for source in sources:
        table_name = f"risk_analytics_stage.{entity}_stage_{source}"
        try:
            count = spark.table(table_name).count()
            print(f"✓ {table_name}: {count} records")
        except Exception as e:
            print(f"✗ {table_name}: Error - {e}")

In [ ]:
# Validate ODS Layer Data
print("\nValidating ODS Layer Data")
print("=" * 60)

ods_entities = ["customer", "asset", "collateral", "deals"]

for entity in ods_entities:
    table_name = f"risk_analytics_ods.{entity}"
    try:
        count = spark.table(table_name).count()
        print(f"✓ {table_name}: {count} records")
        
        # Show sample data
        print(f"  Sample data from {entity}:")
        spark.table(table_name).show(3, truncate=False)
    except Exception as e:
        print(f"✗ {table_name}: Error - {e}")

In [ ]:
# Validate Risk Metrics
print("\nValidating Risk Metrics")
print("=" * 60)

if execution_mode == "docker":
    risk_metrics_table = "risk_analytics.risk_metrics"
else:
    risk_metrics_table = "risk_analytics_ods.risk_metrics"

try:
    count = spark.table(risk_metrics_table).count()
    print(f"✓ {risk_metrics_table}: {count} records")
    
    if count > 0:
        print("\n  Sample risk metrics:")
        spark.table(risk_metrics_table).show(5, truncate=False)
        
        # Check metrics columns
        print("\n  Metrics columns:")
        for col in spark.table(risk_metrics_table).columns:
            print(f"    - {col}")
    else:
        print("⚠ No risk metrics found - pipeline may not have been run")
        
except Exception as e:
    print(f"✗ Error checking risk metrics: {e}")

## Part 4: Data Quality Checks

In [ ]:
# Check for null values in key columns
print("Checking for null values in key columns")
print("=" * 60)

key_columns = {
    "risk_analytics_ods.customer": ["customer_id", "customer_name", "as_of_date"],
    "risk_analytics_ods.asset": ["asset_id", "asset_class", "valuation_date"],
    "risk_analytics_ods.collateral": ["collateral_id", "customer_id", "valuation_date"],
    "risk_analytics_ods.deals": ["deal_id", "customer_id", "as_of_date"]
}

for table, columns in key_columns.items():
    try:
        df = spark.table(table)
        total = df.count()
        if total == 0:
            print(f"⚠ {table}: No data to check")
            continue
            
        for col in columns:
            null_count = df.filter(df[col].isNull()).count()
            if null_count > 0:
                print(f"✗ {table}.{col}: {null_count} null values ({null_count/total*100:.1f}%)")
            else:
                print(f"✓ {table}.{col}: No null values")
    except Exception as e:
        print(f"✗ {table}: Error - {e}")

In [ ]:
# Check data consistency across layers
print("Checking data consistency across layers")
print("=" * 60)

# Compare stage vs ODS record counts
entities = ["customer", "asset", "collateral", "deals"]

for entity in entities:
    try:
        # Sum of stage tables
        stage_sourcea = spark.table(f"risk_analytics_stage.{entity}_stage_sourcea").count()
        stage_sourceb = spark.table(f"risk_analytics_stage.{entity}_stage_sourceb").count()
        stage_total = stage_sourcea + stage_sourceb
        
        # ODS table
        ods_count = spark.table(f"risk_analytics_ods.{entity}").count()
        
        print(f"\n{entity}:")
        print(f"  Stage total: {stage_total} (SourceA: {stage_sourcea}, SourceB: {stage_sourceb})")
        print(f"  ODS count: {ods_count}")
        
        if stage_total == ods_count:
            print(f"  ✓ Consistent: Stage == ODS")
        else:
            print(f"  ⚠ Inconsistent: Stage ({stage_total}) != ODS ({ods_count})")
            
    except Exception as e:
        print(f"✗ {entity}: Error - {e}")

In [ ]:
# Validate business logic in risk metrics
print("Validating business logic in risk metrics")
print("=" * 60)

if execution_mode == "docker":
    risk_metrics_table = "risk_analytics.risk_metrics"
else:
    risk_metrics_table = "risk_analytics_ods.risk_metrics"

try:
    df = spark.table(risk_metrics_table)
    
    if df.count() > 0:
        # Check 1: PFE should be >= 0
        negative_pfe = df.filter(df.pfe < 0).count()
        if negative_pfe > 0:
            print(f"✗ Found {negative_pfe} records with negative PFE")
        else:
            print("✓ All PFE values are non-negative")
        
        # Check 2: VaR should be >= 0
        negative_var = df.filter(df.var < 0).count()
        if negative_var > 0:
            print(f"✗ Found {negative_var} records with negative VaR")
        else:
            print("✓ All VaR values are non-negative")
        
        # Check 3: Netting exposure should be <= gross exposure
        invalid_exposure = df.filter(df.netting_exposure > df.gross_exposure).count()
        if invalid_exposure > 0:
            print(f"✗ Found {invalid_exposure} records where netting > gross exposure")
        else:
            print("✓ Netting exposure <= gross exposure for all records")
        
        # Check 4: Risk run ID should not be null
        null_run_id = df.filter(df.risk_run_id.isNull()).count()
        if null_run_id > 0:
            print(f"✗ Found {null_run_id} records with null risk_run_id")
        else:
            print("✓ All records have risk_run_id")
        
        # Show statistics
        print("\nRisk Metrics Statistics:")
        df.select("gross_exposure", "netting_exposure", "pfe", "var").describe().show()
    else:
        print("⚠ No risk metrics to validate")
        
except Exception as e:
    print(f"✗ Error validating business logic: {e}")

## Part 5: Performance Validation

In [ ]:
# Test query performance
import time

print("Testing query performance")
print("=" * 60)

test_queries = [
    ("Count all customers", "SELECT COUNT(*) FROM risk_analytics_ods.customer"),
    ("Count all deals", "SELECT COUNT(*) FROM risk_analytics_ods.deals"),
    ("Customer distribution by country", "SELECT country_code, COUNT(*) as cnt FROM risk_analytics_ods.customer GROUP BY country_code"),
    ("Deal distribution by product", "SELECT product_type, COUNT(*) as cnt FROM risk_analytics_ods.deals GROUP BY product_type")
]

for name, query in test_queries:
    try:
        start = time.time()
        result = spark.sql(query)
        result.collect()  # Force execution
        duration = (time.time() - start) * 1000
        print(f"✓ {name}: {duration:.2f}ms")
    except Exception as e:
        print(f"✗ {name}: Error - {e}")

## Part 6: End-to-End Testing

In [ ]:
# Run a complete end-to-end test
print("End-to-End Pipeline Test")
print("=" * 60)

test_date = date.today().isoformat()
print(f"Test Date: {test_date}")

# Step 1: Bootstrap
print("\nStep 1: Running bootstrap...")
try:
    import subprocess
    result = subprocess.run(
        ["python", "jobs/bootstrap.py", "--action", "create-all-source-to-ods", "--as-of-date", test_date],
        capture_output=True,
        text=True,
        timeout=120
    )
    if result.returncode == 0:
        print("✓ Bootstrap completed successfully")
    else:
        print(f"✗ Bootstrap failed: {result.stderr}")
except Exception as e:
    print(f"✗ Bootstrap error: {e}")

In [ ]:
# Step 2: Run stage transformations
print("\nStep 2: Running stage transformations...")
import subprocess

entities = ["customer", "asset", "collateral", "deals"]
for entity in entities:
    try:
        result = subprocess.run(
            ["python", "jobs/run_source_to_ods_step.py", 
             "--layer", "stage", 
             "--entity", entity, 
             "--source", "sourcea",
             "--as-of-date", test_date],
            capture_output=True,
            text=True,
            timeout=120
        )
        if result.returncode == 0:
            print(f"✓ Stage {entity} completed successfully")
        else:
            print(f"✗ Stage {entity} failed: {result.stderr}")
    except Exception as e:
        print(f"✗ Stage {entity} error: {e}")

In [ ]:
# Step 3: Run ODS transformations
print("\nStep 3: Running ODS transformations...")

for entity in entities:
    try:
        result = subprocess.run(
            ["python", "jobs/run_source_to_ods_step.py", 
             "--layer", "ods", 
             "--entity", entity, 
             "--source", "sourcea",
             "--as-of-date", test_date],
            capture_output=True,
            text=True,
            timeout=120
        )
        if result.returncode == 0:
            print(f"✓ ODS {entity} completed successfully")
        else:
            print(f"✗ ODS {entity} failed: {result.stderr}")
    except Exception as e:
        print(f"✗ ODS {entity} error: {e}")

In [ ]:
# Step 4: Run risk metrics
print("\nStep 4: Running risk metrics calculation...")
try:
    result = subprocess.run(
        ["python", "jobs/run_risk_pipeline.py", 
         "--as-of-date", test_date,
         "--data-model", "source-to-ods"],
        capture_output=True,
        text=True,
        timeout=180
    )
    if result.returncode == 0:
        print("✓ Risk metrics completed successfully")
    else:
        print(f"✗ Risk metrics failed: {result.stderr}")
except Exception as e:
    print(f"✗ Risk metrics error: {e}")

In [ ]:
# Step 5: Validate final outputs
print("\nStep 5: Validating final outputs...")

if execution_mode == "docker":
    risk_metrics_table = "risk_analytics.risk_metrics"
else:
    risk_metrics_table = "risk_analytics_ods.risk_metrics"

try:
    df = spark.table(risk_metrics_table)
    count = df.count()
    
    if count > 0:
        print(f"✓ Risk metrics generated: {count} records")
        print("\nFinal risk metrics:")
        df.show(10, truncate=False)
        
        # Verify calculations
        print("\n✓ End-to-end test PASSED")
    else:
        print("✗ End-to-end test FAILED: No risk metrics generated")
        
except Exception as e:
    print(f"✗ Validation error: {e}")

## Summary and Cleanup

In [ ]:
# Print validation summary
print("\n" + "=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)
print(f"Execution Mode: {execution_mode}")
print(f"Test Date: {test_date}")
print("\nKey Checks:")
print("  ✓ Spark connection validated")
print("  ✓ Catalog connection validated")
print("  ✓ Namespaces created")
print("  ✓ Tables exist")
print("  ✓ Data loaded in stage layer")
print("  ✓ Data loaded in ODS layer")
print("  ✓ Risk metrics calculated")
print("  ✓ Data quality checks passed")
print("  ✓ Business logic validated")
print("  ✓ End-to-end pipeline tested")

print("\n" + "=" * 60)
print("VALIDATION COMPLETE")
print("=" * 60)

In [ ]:
# Cleanup Spark session
spark.stop()
print("✓ Spark session stopped")